# W04D03 — Hyperparameter Tuning: GridSearchCV
**ML Engineer Journey | Phase 1 — ML Fundamentals | Week 4 Day 3**

---

## Inti dari Topic Ini (Baca ini dulu sebelum apapun)

> **GridSearchCV = coba semua kombinasi parameter secara sistematis, evaluasi tiap kombinasi pakai cross-validation, pilih yang terbaik.**

Tiga hal yang wajib kamu ingat dari sesi ini:

1. **Grid besar = waktu eksponensial** — tambah 1 parameter saja bisa 3× lipat total model yang dilatih
2. **scoring='roc_auc', bukan 'accuracy'** — accuracy bergantung threshold 0.5 yang tidak selalu optimal
3. **RF lebih lambat dari XGBoost di dataset kecil BUKAN karena sequential vs parallel** — tapi karena overhead setup parallelisme RF tidak sepadan untuk dataset kecil

---

## Struktur Notebook

| # | Section | Isi |
|---|---------|-----|
| 1 | Setup & Data | Load dataset, split train/test |
| 2 | GridSearch RF | Implementasi + catat waktu |
| 3 | GridSearch XGBoost | Implementasi + catat waktu |
| 4 | Analisa cv_results_ | Train vs val gap, deteksi overfit |
| 5 | Evaluasi Test Set | Classification report kedua model |
| 6 | Koreksi Miskonsepsi | Penjelasan mendalam hal yang sering salah |
| 7 | Ringkasan & Checklist | Before you run GridSearch |


---
## Section 1 — Setup & Data

Kita pakai **breast cancer dataset** dari sklearn:
- 569 sampel, 30 fitur
- Target: 0 (malignant/ganas) vs 1 (benign/jinak)
- Dataset kecil tapi representatif untuk eksperimen GridSearch

> **Kenapa dataset ini?** Ukurannya cukup kecil untuk GridSearch selesai dalam hitungan menit,
> tapi cukup bermakna untuk menunjukkan perbedaan antar model.


In [1]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Shape   : {X.shape}")
print(f"Kelas   : {dict(zip(data.target_names, np.bincount(y)))}")
print(f"Fitur   : {data.feature_names[:5]} ... ({X.shape[1]} total)")

# Train-test split — stratify supaya rasio kelas sama di train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # PENTING: tanpa ini distribusi kelas bisa tidak seimbang
)

print(f"\nTrain  : {X_train.shape[0]} sampel")
print(f"Test   : {X_test.shape[0]} sampel")
print(f"Rasio test: {X_test.shape[0]/X.shape[0]:.0%}")


Shape   : (569, 30)
Kelas   : {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}
Fitur   : ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness'] ... (30 total)

Train  : 455 sampel
Test   : 114 sampel
Rasio test: 20%


---
## Section 2 — GridSearch: Random Forest

### Hitung dulu sebelum run — ini WAJIB

Sebelum menjalankan GridSearch, hitung total model yang akan dilatih:

```
param_grid: 3 nilai × 3 nilai × 2 nilai = 18 kombinasi
cv = 5 fold
Total model = 18 × 5 = 90 model
```

Ini masih dalam batas wajar. Kalau kamu tambah satu parameter lagi dengan 4 nilai:
**18 × 4 × 5 = 360 model** — mulai mahal.

### Parameter yang dipilih dan alasannya

| Parameter | Nilai yang Dicoba | Alasan |
|-----------|-----------------|--------|
| `n_estimators` | 100, 200, 300 | Tradeoff jumlah pohon vs waktu |
| `max_depth` | 3, 5, None | None = pohon tumbuh sampai pure — risiko overfit |
| `min_samples_split` | 2, 5 | Kontrol minimum sampel untuk split node |

> **Catatan:** `max_depth=None` di RF artinya pohon tumbuh tanpa batas kedalaman.
> Ini sering menghasilkan train AUC mendekati 1.0 tapi val AUC lebih rendah — overfit.
> GridSearch akan "menyeleksi" ini kalau CV score-nya memang lebih buruk.


In [2]:
# ── Parameter Grid ───────────────────────────────────────────
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, None],       # None = tidak ada batas kedalaman
    'min_samples_split': [2, 5],     # minimum sampel untuk split node
}

# Hitung total model sebelum run
from itertools import product
total_combos = 1
for v in param_grid_rf.values():
    total_combos *= len(v)
print(f"Total kombinasi : {total_combos}")
print(f"Total model (×5 fold): {total_combos * 5}")


Total kombinasi : 18
Total model (×5 fold): 90


In [3]:
# ── GridSearchCV RF ──────────────────────────────────────────
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv=5,                    # 5-fold cross-validation
    scoring='roc_auc',       # BUKAN accuracy — lihat Section 6 untuk penjelasan
    n_jobs=-1,               # parallelisme di level CV
    verbose=1,
    return_train_score=True  # WAJIB — untuk deteksi gap train vs val
)

start_rf = time.time()
grid_rf.fit(X_train, y_train)
rf_time = time.time() - start_rf

print(f"\n{'='*45}")
print(f"Waktu GridSearch RF  : {rf_time:.1f} detik")
print(f"Best params RF       : {grid_rf.best_params_}")
print(f"Best CV AUC RF       : {grid_rf.best_score_:.4f}")
print(f"{'='*45}")


Fitting 5 folds for each of 18 candidates, totalling 90 fits

Waktu GridSearch RF  : 67.8 detik
Best params RF       : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
Best CV AUC RF       : 0.9889


---
## Section 3 — GridSearch: XGBoost

Grid yang sama ukurannya (18 kombinasi × 5 fold = 90 model).
Kita akan bandingkan waktu komputasinya dengan RF.

### Kenapa max_depth XGBoost lebih kecil dari RF?

Di boosting, pohon yang dalam berbahaya:
- Tree overfits ke residual satu iterasi
- Tree berikutnya mengoreksi **noise**, bukan pola
- Hasil: model kompleks yang hafal outlier

Makanya `max_depth` di XGBoost biasanya 3–7, jarang lebih dari 10.
Di RF, pohon dalam tidak masalah karena **averaging** dari banyak pohon
meredam overfit individu.


In [4]:
# ── Parameter Grid XGBoost ──────────────────────────────────
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
}

total_combos_xgb = 1
for v in param_grid_xgb.values():
    total_combos_xgb *= len(v)
print(f"Total kombinasi : {total_combos_xgb}")
print(f"Total model (×5 fold): {total_combos_xgb * 5}")


Total kombinasi : 18
Total model (×5 fold): 90


In [5]:
# ── GridSearchCV XGBoost ────────────────────────────────────
xgb_base = xgb.XGBClassifier(
    random_state=42,
    eval_metric='auc',
    verbosity=0
)

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

start_xgb = time.time()
grid_xgb.fit(X_train, y_train)
xgb_time = time.time() - start_xgb

print(f"\n{'='*45}")
print(f"Waktu GridSearch XGB : {xgb_time:.1f} detik")
print(f"Best params XGBoost  : {grid_xgb.best_params_}")
print(f"Best CV AUC XGBoost  : {grid_xgb.best_score_:.4f}")
print(f"{'='*45}")


Fitting 5 folds for each of 18 candidates, totalling 90 fits

Waktu GridSearch XGB : 10.6 detik
Best params XGBoost  : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
Best CV AUC XGBoost  : 0.9904


In [6]:
# ── Perbandingan Cepat ──────────────────────────────────────
print("\n⏱ PERBANDINGAN WAKTU")
print(f"{'RF GridSearch':20}: {rf_time:.1f}s")
print(f"{'XGB GridSearch':20}: {xgb_time:.1f}s")
print(f"{'RF lebih lambat':20}: {rf_time/xgb_time:.1f}x")

print("\n📊 PERBANDINGAN CV AUC")
print(f"{'RF best CV AUC':20}: {grid_rf.best_score_:.4f}")
print(f"{'XGB best CV AUC':20}: {grid_xgb.best_score_:.4f}")
winner = 'XGBoost' if grid_xgb.best_score_ > grid_rf.best_score_ else 'RF'
print(f"{'Unggul':20}: {winner} (selisih {abs(grid_xgb.best_score_ - grid_rf.best_score_):.4f})")



⏱ PERBANDINGAN WAKTU
RF GridSearch       : 67.8s
XGB GridSearch      : 10.6s
RF lebih lambat     : 6.4x

📊 PERBANDINGAN CV AUC
RF best CV AUC      : 0.9889
XGB best CV AUC     : 0.9904
Unggul              : XGBoost (selisih 0.0015)


---
## Section 4 — Analisa cv_results_: Train vs Val Gap

`cv_results_` menyimpan **semua histori eksperimen** GridSearch — setiap kombinasi,
mean score, std score, waktu. Ini goldmine untuk debugging.

### Yang perlu diperhatikan:

| Kolom | Artinya |
|-------|---------|
| `mean_test_score` | Rata-rata CV AUC dari 5 fold — **angka yang paling penting** |
| `mean_train_score` | Rata-rata train AUC dari 5 fold |
| `std_test_score` | Standar deviasi CV score — tinggi = model tidak stabil |
| Gap (train - test) | Indikator overfit — lebih besar lebih buruk |

### ⚠️ Koreksi Penting: Gap Kecil ≠ Tidak Overfit Sama Sekali

Banyak yang mengira gap kecil artinya model tidak overfit. **Ini tidak tepat.**

- **Train AUC 0.9997** = model hampir hafal sempurna data training → ada overfit
- **CV AUC 0.9889** = generalise dengan baik → masih acceptable
- **Gap 0.0108** = overfit **ringan yang masih dalam batas toleransi** (threshold aman < 0.02)

GridSearch sudah "menyelamatkan" kita dengan memilih `max_depth=5` bukan `None`.
Tapi train AUC yang mendekati 1.0 tetap perlu diwaspadai.


In [7]:
# ── Analisa cv_results_ RF ──────────────────────────────────
results_rf = pd.DataFrame(grid_rf.cv_results_)
results_rf['gap'] = results_rf['mean_train_score'] - results_rf['mean_test_score']

cols = ['params', 'mean_test_score', 'mean_train_score', 'std_test_score', 'gap']
top5_rf = results_rf.nlargest(5, 'mean_test_score')[cols].reset_index(drop=True)

print("Top 5 Kombinasi RF (by CV AUC):")
print(top5_rf.to_string())


Top 5 Kombinasi RF (by CV AUC):
                                                             params  mean_test_score  mean_train_score  std_test_score       gap
0     {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}         0.988854          0.999703        0.010374  0.010849
1     {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}         0.988235          0.999723        0.012578  0.011487
2     {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 300}         0.987719          0.999742        0.013362  0.012023
3     {'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 200}         0.987513          0.996840        0.010296  0.009327
4  {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}         0.987513          1.000000        0.012851  0.012487


In [8]:
# ── Detail Kombinasi Terbaik RF ─────────────────────────────
best_idx_rf = grid_rf.best_index_
train_auc_rf = grid_rf.cv_results_['mean_train_score'][best_idx_rf]
val_auc_rf   = grid_rf.best_score_
gap_rf       = train_auc_rf - val_auc_rf

print("RF — Kombinasi Terbaik:")
print(f"  Params     : {grid_rf.best_params_}")
print(f"  Train AUC  : {train_auc_rf:.4f}")
print(f"  CV AUC     : {val_auc_rf:.4f}")
print(f"  Gap        : {gap_rf:.4f}", end=" ")

if gap_rf < 0.01:
    print("✅ Sangat kecil — generalise baik")
elif gap_rf < 0.02:
    print("⚠️  Overfit ringan — masih acceptable")
else:
    print("❌ Overfit signifikan — perlu regularisasi lebih")

print()
print("INTERPRETASI YANG BENAR:")
print(f"  Train AUC {train_auc_rf:.4f} = model sedikit hafal data training")
print(f"  Gap {gap_rf:.4f} = overfit ringan, tapi MASIH ADA overfit")
print(f"  GridSearch memilih max_depth=5 karena CV score-nya lebih baik")
print(f"  dari max_depth=None yang pasti punya gap lebih besar")


RF — Kombinasi Terbaik:
  Params     : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
  Train AUC  : 0.9997
  CV AUC     : 0.9889
  Gap        : 0.0108 ⚠️  Overfit ringan — masih acceptable

INTERPRETASI YANG BENAR:
  Train AUC 0.9997 = model sedikit hafal data training
  Gap 0.0108 = overfit ringan, tapi MASIH ADA overfit
  GridSearch memilih max_depth=5 karena CV score-nya lebih baik
  dari max_depth=None yang pasti punya gap lebih besar


In [9]:
# ── Detail Kombinasi Terbaik XGBoost ────────────────────────
best_idx_xgb = grid_xgb.best_index_
train_auc_xgb = grid_xgb.cv_results_['mean_train_score'][best_idx_xgb]
val_auc_xgb   = grid_xgb.best_score_
gap_xgb       = train_auc_xgb - val_auc_xgb

print("XGBoost — Kombinasi Terbaik:")
print(f"  Params     : {grid_xgb.best_params_}")
print(f"  Train AUC  : {train_auc_xgb:.4f}")
print(f"  CV AUC     : {val_auc_xgb:.4f}")
print(f"  Gap        : {gap_xgb:.4f}", end=" ")

if gap_xgb < 0.01:
    print("✅ Sangat kecil")
elif gap_xgb < 0.02:
    print("⚠️  Overfit ringan — acceptable")
else:
    print("❌ Overfit signifikan")


XGBoost — Kombinasi Terbaik:
  Params     : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
  Train AUC  : 1.0000
  CV AUC     : 0.9904
  Gap        : 0.0096 ✅ Sangat kecil


---
## Section 5 — Evaluasi di Test Set

### Aturan utama: test set disentuh SEKALI, di akhir, setelah semua keputusan dibuat

`best_estimator_` sudah di-refit menggunakan `best_params_` ke **seluruh X_train**
(bukan hanya satu fold). Langsung bisa dipakai untuk prediksi.

### ⚠️ Peringatan: Jangan Terlalu Percaya Test AUC dari Dataset Kecil

Test set kita hanya **60 sampel**. Dengan dataset sekecil ini:
- Satu atau dua prediksi yang kebetulan benar bisa mendongkrak AUC secara artifisial
- Test AUC bisa lebih tinggi dari CV AUC — bukan karena model lebih baik,
  tapi karena **keberuntungan statistik**
- **CV AUC adalah estimasi yang lebih jujur** karena dihitung dari lebih banyak data


In [10]:
# ── Evaluasi RF di Test Set ─────────────────────────────────
best_rf = grid_rf.best_estimator_

y_pred_rf  = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
test_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("=" * 50)
print("RANDOM FOREST — Test Set Report")
print("=" * 50)
print(classification_report(y_test, y_pred_rf, target_names=['Malignant', 'Benign']))
print(f"Test AUC  : {test_auc_rf:.4f}")
print(f"CV AUC    : {grid_rf.best_score_:.4f}")

if test_auc_rf > grid_rf.best_score_:
    diff = test_auc_rf - grid_rf.best_score_
    print(f"\n⚠️  Test AUC LEBIH TINGGI dari CV AUC (selisih {diff:.4f})")
    print("   Ini bukan berarti model luar biasa.")
    print("   Test set hanya 60 sampel — angka bisa kebetulan tinggi.")
    print("   CV AUC tetap estimasi yang lebih reliable.")


RANDOM FOREST — Test Set Report
              precision    recall  f1-score   support

   Malignant       0.95      0.93      0.94        42
      Benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Test AUC  : 0.9934
CV AUC    : 0.9889

⚠️  Test AUC LEBIH TINGGI dari CV AUC (selisih 0.0045)
   Ini bukan berarti model luar biasa.
   Test set hanya 60 sampel — angka bisa kebetulan tinggi.
   CV AUC tetap estimasi yang lebih reliable.


In [11]:
# ── Evaluasi XGBoost di Test Set ────────────────────────────
best_xgb = grid_xgb.best_estimator_

y_pred_xgb  = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
test_auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print("=" * 50)
print("XGBOOST — Test Set Report")
print("=" * 50)
print(classification_report(y_test, y_pred_xgb, target_names=['Malignant', 'Benign']))
print(f"Test AUC  : {test_auc_xgb:.4f}")
print(f"CV AUC    : {grid_xgb.best_score_:.4f}")


XGBOOST — Test Set Report
              precision    recall  f1-score   support

   Malignant       0.97      0.90      0.94        42
      Benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Test AUC  : 0.9917
CV AUC    : 0.9904


In [12]:
# ── Tabel Perbandingan Final ─────────────────────────────────
print("\n" + "=" * 60)
print("PERBANDINGAN FINAL")
print("=" * 60)
print(f"{'Metrik':<25} {'Random Forest':>15} {'XGBoost':>15}")
print("-" * 60)
print(f"{'CV AUC (reliable)':<25} {grid_rf.best_score_:>15.4f} {grid_xgb.best_score_:>15.4f}")
print(f"{'Test AUC (60 sampel)':<25} {test_auc_rf:>15.4f} {test_auc_xgb:>15.4f}")
print(f"{'Train AUC':<25} {grid_rf.cv_results_["mean_train_score"][grid_rf.best_index_]:>15.4f} {grid_xgb.cv_results_["mean_train_score"][grid_xgb.best_index_]:>15.4f}")
print(f"{'Waktu GridSearch':<25} {rf_time:>14.1f}s {xgb_time:>14.1f}s")
print(f"{'Total kombinasi':<25} {len(grid_rf.cv_results_["params"]):>15} {len(grid_xgb.cv_results_["params"]):>15}")
print("=" * 60)



PERBANDINGAN FINAL
Metrik                      Random Forest         XGBoost
------------------------------------------------------------
CV AUC (reliable)                  0.9889          0.9904
Test AUC (60 sampel)               0.9934          0.9917
Train AUC                          0.9997          1.0000
Waktu GridSearch                    67.8s           10.6s
Total kombinasi                        18              18


---
## Section 6 — Koreksi Miskonsepsi (Baca Ini Baik-Baik)

Ini bagian yang paling penting dari notebook ini.
Tiga konsep yang sering dipahami keliru.

---

### Miskonsepsi #1: "XGBoost lebih cepat karena parallel, RF lebih lambat karena sequential"

**❌ SALAH — ini terbalik.**

| | Random Forest | XGBoost |
|--|--|--|
| **Cara kerja pohon** | **PARALEL** — 300 pohon dilatih sekaligus | **SEQUENTIAL** — boosting, satu pohon setelah pohon sebelumnya |
| **Kenapa RF lebih lambat di data kecil?** | Overhead setup parallelisme tidak sepadan | Pohon dangkal, overhead kecil |

**✅ PENJELASAN YANG BENAR:**

RF lebih lambat di dataset 569 baris karena:
1. **Overhead parallelisme lebih mahal dari training itu sendiri** — Python harus spawn threads, copy data ke tiap core CPU, sync hasilnya
2. **XGBoost pohonnya dangkal** (max_depth 3–7) — setiap pohon selesai dalam milidetik, meski sequential
3. **Dataset kecil muat sepenuhnya di memori** — XGBoost bisa akses langsung tanpa distribusi ke banyak proses

**Analogi:** Menyuruh 300 orang mengupas 1 bawang masing-masing vs 1 orang mengupas 300 bawang berturut-turut. Untuk bawang kecil, overhead koordinasi 300 orang justru lebih lama.

---

### Miskonsepsi #2: "Gap kecil antara train dan val berarti tidak overfit sama sekali"

**❌ KURANG TEPAT.**

Gap 0.0108 dengan Train AUC 0.9997 berarti:
- Model **masih overfit ringan** — ia hampir hafal data training
- Gap kecil artinya overfit-nya **masih dalam batas toleransi**, bukan tidak ada overfit
- Threshold aman: gap < 0.02 untuk dataset ini

**✅ FORMULASI YANG BENAR:**
> "Gap 0.0108 menunjukkan overfit ringan yang masih acceptable.
> GridSearch menyelamatkan kita dari pilihan yang lebih buruk (max_depth=None),
> tapi train AUC yang mendekati 1.0 tetap sinyal bahwa model sedikit menghafal."

---

### Miskonsepsi #3: "Test AUC lebih tinggi dari CV AUC berarti model sangat bagus"

**❌ HATI-HATI.**

Test AUC RF = 0.9997, CV AUC = 0.9889. Test AUC lebih tinggi?

Ini bukan prestasi — ini **artefak statistik dari sampel kecil**.
Test set kita hanya 60 sampel. Dengan 60 sampel:
- Satu prediksi berbeda = perubahan AUC ~0.01
- Hasil bisa fluktuatif tergantung random state split

**CV AUC dihitung dari semua data training via 5 fold = lebih banyak sampel = lebih reliable.**

> **Aturan:** Kalau test set < 200 sampel, jangan terlalu percaya angkanya. CV AUC adalah estimasi performa yang lebih jujur.


---
## Section 7 — Ringkasan & Checklist

### Kapan pakai GridSearch vs RandomizedSearch?

| Kondisi | Pilihan |
|---------|---------|
| Total kombinasi ≤ 150 | GridSearch — exhaustive, pasti menemukan yang terbaik dalam grid |
| Total kombinasi > 150 | RandomizedSearch — sampling acak, lebih efisien |
| Budget waktu terbatas | RandomizedSearch dengan `n_iter` yang terkontrol |
| Parameter space kontinu | RandomizedSearch dengan distribusi (scipy.stats) |

---

### Checklist Sebelum Menjalankan GridSearch

- [ ] **Hitung total model dulu** — kombinasi × fold. Kalau > 200, pertimbangkan RandomizedSearch
- [ ] **scoring='roc_auc'** atau metrik yang sesuai konteks — bukan accuracy secara default
- [ ] **return_train_score=True** — untuk bisa deteksi gap train vs val
- [ ] **Catat waktu komputasi** — pakai `time.time()` sebelum dan sesudah `.fit()`
- [ ] **Test set disentuh SEKALI** — hanya di akhir setelah semua keputusan selesai
- [ ] **Interpretasi gap dengan benar** — gap kecil = overfit masih tolerable, bukan tidak ada overfit
- [ ] **Jangan terlalu percaya test AUC dari dataset kecil** — CV AUC lebih reliable

---

### Kata Kunci untuk Review Cepat

```
GridSearchCV      = exhaustive search semua kombinasi × CV fold
best_estimator_   = model terbaik, sudah di-refit ke seluruh X_train
best_score_       = CV AUC terbaik (bukan test set)
cv_results_       = semua histori eksperimen — untuk analisa mendalam
return_train_score = wajib True untuk deteksi overfit
scoring           = pakai 'roc_auc', bukan 'accuracy'
```

---

### Next: W04D04 — RandomizedSearch + Early Stopping

GridSearch exhaustive tapi mahal untuk grid besar.
RandomizedSearch sampling kombinasi secara acak dengan `n_iter` yang terkontrol —
dikombinasi early stopping supaya XGBoost berhenti sebelum overfit.


In [13]:
# ── Quick Summary ────────────────────────────────────────────
print("=" * 60)
print("RINGKASAN SESI W04D03")
print("=" * 60)
print(f"\nRandom Forest:")
print(f"  Best params  : {grid_rf.best_params_}")
print(f"  CV AUC       : {grid_rf.best_score_:.4f}")
print(f"  Waktu        : {rf_time:.1f}s")

print(f"\nXGBoost:")
print(f"  Best params  : {grid_xgb.best_params_}")
print(f"  CV AUC       : {grid_xgb.best_score_:.4f}")
print(f"  Waktu        : {xgb_time:.1f}s")

print(f"\nKesimpulan:")
print(f"  - XGBoost unggul tipis di CV AUC")
print(f"  - XGBoost {rf_time/xgb_time:.1f}x lebih cepat karena overhead parallelisme RF")
print(f"    tidak sepadan untuk dataset {X.shape[0]} baris")
print(f"  - CV AUC lebih reliable daripada test AUC (60 sampel terlalu kecil)")
print(f"  - Gap train-val RF {grid_rf.cv_results_['mean_train_score'][grid_rf.best_index_] - grid_rf.best_score_:.4f} = overfit ringan, masih acceptable")


RINGKASAN SESI W04D03

Random Forest:
  Best params  : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
  CV AUC       : 0.9889
  Waktu        : 67.8s

XGBoost:
  Best params  : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
  CV AUC       : 0.9904
  Waktu        : 10.6s

Kesimpulan:
  - XGBoost unggul tipis di CV AUC
  - XGBoost 6.4x lebih cepat karena overhead parallelisme RF
    tidak sepadan untuk dataset 569 baris
  - CV AUC lebih reliable daripada test AUC (60 sampel terlalu kecil)
  - Gap train-val RF 0.0108 = overfit ringan, masih acceptable
